**historical stock analysis - assignment 3 proposed analysis**

this notebook builds the new charts for assignment 3. part a checks h1, whether the ai rally is really just a handful of chip stocks. part b checks h2 and h3 together, whether stocks from different regions move more alike specifically during a crash, and whether volume and volatility spike at the same time.

both parts build on the assignment 2 notebook, especially the open question at the end of part 5 there, does the correlation pattern in figure 5 actually hold up during a real crisis, or does it just look that way because calm and volatile years got averaged together over the full 2006 to 2026 history.

**step 1: load the data and work out daily returns**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/stock_data.csv", parse_dates=["Date"])
df = df.sort_values(["Ticker", "Date"])
df["Daily Return"] = df.groupby("Ticker")["Adj Close"].pct_change()
df.head()

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Daily Return
0,2006-01-02,000660.KS,36100.0,37650.0,35850.0,37600.0,27358.396484,13334311.0,NaN
1,2006-01-03,000660.KS,38100.0,38400.0,36850.0,38200.0,27794.970703,13815168.0,0.015958
2,2006-01-04,000660.KS,38250.0,39050.0,35300.0,35400.0,25757.650391,23064896.0,-0.073298
3,2006-01-05,000660.KS,35900.0,36000.0,33600.0,34750.0,25284.687500,21043236.0,-0.018362
4,2006-01-06,000660.KS,34950.0,35850.0,34850.0,35100.0,25539.355469,12557226.0,0.010072


**part a: is the ai rally really just a few chip stocks (h1)**

Assignment 2's Figure 8 already showed nvidia and broadcom pulling far ahead of asml, tsm, apple, and microsoft since 2009. here we check the same idea a different way, specifically for the period since january 2023, when the ai rally is supposed to have started.

instead of looking at single stocks like Figure 8 did, we split them into two groups, a chip group (nvda, avgo, tsm, asml, amd, mu, lrcx) and a rest of tech group (aapl, msft, googl, amzn, meta, nflx, orcl, csco), index both to 100 at the same start date, and average each group into one line so the two groups can be compared directly.

In [2]:
chip_group = ["NVDA", "AVGO", "TSM", "ASML", "AMD", "MU", "LRCX"]
rest_group = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NFLX", "ORCL", "CSCO"]

start_date = "2023-01-03"
recent = df[df["Date"] >= start_date]

prices = recent.pivot(index="Date", columns="Ticker", values="Adj Close")
indexed = prices / prices.iloc[0] * 100

chip_line = indexed[chip_group].mean(axis=1)
rest_line = indexed[rest_group].mean(axis=1)

print(f"chip group ends at: {chip_line.iloc[-1]:.0f}")
print(f"rest of tech group ends at: {rest_line.iloc[-1]:.0f}")

chip group ends at: 649
rest of tech group ends at: 268


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(chip_line.index, chip_line.values, color="#2a78d6", linewidth=2, label="chip / AI group average")
ax.plot(rest_line.index, rest_line.values, color="#eb6834", linewidth=2, label="rest of tech group average")

ax.set_title("Figure A: chip/AI stocks vs rest of tech, indexed to 100 on 2023-01-03")
ax.set_xlabel("Date")
ax.set_ylabel("Indexed price, 100 = value on 2023-01-03")
ax.legend()
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.figtext(
    0.01, -0.04,
    "Source: data/stock_data.csv, 2023-01-03 to 2026-02-20. Chip/AI group: NVDA, AVGO, TSM, ASML, AMD, MU, LRCX.\n"
    "Rest of tech group: AAPL, MSFT, GOOGL, AMZN, META, NFLX, ORCL, CSCO. Each group line is a simple average of the indexed prices.",
    fontsize=8,
)
plt.tight_layout()
plt.savefig("assignments/figures/assignment3/figure_a_ai_rally.png", dpi=150, bbox_inches="tight")
plt.show()

- the chip group average and the rest of tech group average start at the same place, 100, on 2023-01-03, and end up nowhere close together

- the gap keeps widening through the whole period instead of happening in one early jump, so this is not a one time event, the chip group kept pulling ahead the whole way

- this backs up h1 using a different method than assignment 2's Figure 8. that figure looked at four individual chip stocks since 2009, this one looks at two whole groups averaged together since 2023 specifically, and both point the same way

**part b: does diversification actually break down in a crash (h2 and h3)**

Assignment 2's Figure 5 built one correlation heatmap using the whole 2006 to 2026 history, and the writeup flagged an open question, does that correlation pattern actually hold up during a real crisis, or does it just look that way because calm and volatile years got averaged together.

here we pick one stock per region, us (jpm), korea (005930.ks), hong kong (0700.hk), switzerland (novn.sw), and paris (mc.pa). this is the same kind of single stock per region assignment 2 already used for lvmh representing "the paris market". we compare four time windows, a calm stretch (2015 to 2017), the 2008 financial crisis, the 2020 covid crash, and the 2022 drawdown. for each window we work out three things, how correlated the five regions are with each other, how volatile they are, and how much more they are trading than normal.

In [4]:
region_ticker = {
    "US": "JPM",
    "Korea": "005930.KS",
    "Hong Kong": "0700.HK",
    "Switzerland": "NOVN.SW",
    "Paris": "MC.PA",
}
tickers = list(region_ticker.values())

periods = {
    "calm\n(2015-2017)": ("2015-01-01", "2017-12-31"),
    "2008\ncrisis": ("2008-09-01", "2009-03-31"),
    "2020 covid\ncrash": ("2020-02-15", "2020-04-30"),
    "2022\ndrawdown": ("2022-01-01", "2022-12-31"),
}

# every possible pair among our 5 region tickers. since each ticker is a different
# region, any pair on this list is automatically a cross region pair
pairs = [
    ("JPM", "005930.KS"), ("JPM", "0700.HK"), ("JPM", "NOVN.SW"), ("JPM", "MC.PA"),
    ("005930.KS", "0700.HK"), ("005930.KS", "NOVN.SW"), ("005930.KS", "MC.PA"),
    ("0700.HK", "NOVN.SW"), ("0700.HK", "MC.PA"),
    ("NOVN.SW", "MC.PA"),
]

returns_wide = df[df["Ticker"].isin(tickers)].pivot(index="Date", columns="Ticker", values="Daily Return")
volume_wide = df[df["Ticker"].isin(tickers)].pivot(index="Date", columns="Ticker", values="Volume")

returns_wide.head()

Ticker,005930.KS,0700.HK,JPM,MC.PA,NOVN.SW
Date,,,,,
2006-01-02,NaN,NaN,NaN,NaN,NaN
2006-01-03,0.006061,NaN,NaN,0.002631,-0.005793
2006-01-04,0.049699,0.040230,-0.005772,0.013779,0.012382
2006-01-05,-0.025825,-0.016575,0.003029,-0.001941,-0.004331
2006-01-06,0.007364,0.056180,0.007046,0.000000,0.005072


In [5]:
# average daily volume during the calm period, one number per ticker
# every other period gets compared back to this baseline
baseline_start, baseline_end = periods["calm\n(2015-2017)"]
baseline_volume = volume_wide.loc[baseline_start:baseline_end].mean()
baseline_volume

Ticker
005930.KS    1.182328e+07
0700.HK      2.093878e+07
JPM          1.562689e+07
MC.PA        8.147078e+05
NOVN.SW      6.770425e+06
dtype: float64

In [6]:
results = []

for period_name, (start, end) in periods.items():
    period_returns = returns_wide.loc[start:end]
    corr_matrix = period_returns.corr()

    pair_correlations = []
    for ticker_a, ticker_b in pairs:
        pair_correlations.append(corr_matrix.loc[ticker_a, ticker_b])
    avg_correlation = sum(pair_correlations) / len(pair_correlations)

    ticker_volatilities = []
    for ticker in tickers:
        ticker_volatilities.append(period_returns[ticker].std())
    avg_volatility = sum(ticker_volatilities) / len(ticker_volatilities)

    period_volume = volume_wide.loc[start:end].mean()
    volume_ratio_per_ticker = period_volume / baseline_volume
    avg_volume_ratio = volume_ratio_per_ticker.mean()

    results.append({
        "period": period_name.replace("\n", " "),
        "avg_correlation": avg_correlation,
        "avg_volatility": avg_volatility,
        "avg_volume_ratio": avg_volume_ratio,
    })

results_table = pd.DataFrame(results)
results_table

,period,avg_correlation,avg_volatility,avg_volume_ratio
0,calm (2015-2017),0.250209,0.015062,1.000000
1,2008 crisis,0.250385,0.045518,2.706844
2,2020 covid crash,0.526433,0.035688,1.680648
3,2022 drawdown,0.209277,0.019800,0.925964


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

bar_color = "#2a78d6"
period_labels = list(periods.keys())

axes[0].bar(period_labels, results_table["avg_correlation"], color=bar_color)
axes[0].set_title("Avg. correlation\nbetween regions")
axes[0].set_ylabel("Correlation (unitless)")

axes[1].bar(period_labels, results_table["avg_volatility"], color=bar_color)
axes[1].set_title("Avg. volatility\n(std of daily return)")
axes[1].set_ylabel("Volatility (unitless)")

axes[2].bar(period_labels, results_table["avg_volume_ratio"], color=bar_color)
axes[2].axhline(1.0, color="#898781", linestyle="--", linewidth=1)
axes[2].set_title("Avg. trading volume\n(vs. calm baseline)")
axes[2].set_ylabel("Times normal volume")

for ax in axes:
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Figure B: correlation, volatility, and volume across four time periods", fontsize=12)
plt.figtext(
    0.01, -0.06,
    "Source: data/stock_data.csv. Region tickers: JPM (US), 005930.KS (Korea), 0700.HK (Hong Kong), NOVN.SW (Switzerland), MC.PA (Paris).\n"
    "Volume ratio is each period's average volume divided by that same ticker's 2015-2017 calm baseline average, then averaged across tickers.",
    fontsize=8,
)
plt.tight_layout()
plt.savefig("assignments/figures/assignment3/figure_b_crisis_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

- during the calm baseline (2015 to 2017), the five regions have an average correlation of about 0.25, some tendency to move together but far from all the way

- the 2020 covid crash shows the clearest jump by far, average correlation more than doubles to about 0.53

- 2008 is surprisingly close to the calm number with this method, about 0.25 either way, barely different. this is probably a limitation of using only one stock per region rather than proof that 2008 was not a real panic. jpm's own trading volume in 2008 was over 5 times its normal level by itself, the highest of any ticker in any period here, which makes sense since jpmorgan acquired bear stearns in march 2008 right in the middle of the crisis. a stock that busy with its own news may not move in step with unrelated regions the same way a calmer stock would, so some of what should be a regional story is probably getting mixed up with a jpm specific one. this is exactly why section 6 lists widening the region basket as the next step

- volume and volatility tell a cleaner story than correlation for 2008. average trading volume was 2.7 times the calm baseline in 2008 and 1.7 times in 2020, while 2022 was actually slightly below normal at 0.9 times. volatility follows the same shape, about three times the calm baseline in 2008, over two times in 2020, and only mildly higher in 2022

- so h3 (volume and volatility spiking specifically during panic driven crashes) holds up clearly across all three periods. h2 (correlation specifically) is clean for 2020 and 2022 but noisier for 2008 with just one stock per region, which is a more honest, if less tidy, result than assuming every crisis looks identical

- this still answers assignment 2's open question, the full history correlation number in Figure 5 is not the whole story, 2020 and 2022 behave very differently from each other and from the long run average, even if 2008 needs a wider basket of stocks to show up clearly in correlation terms